# AQUI SE FILTRA EL DATO Y TIEMPO QUE NECESITAMOS

In [ ]:
import pandas as pd

RUTA_ENTRADA = r"C:\Users\manue\Downloads\3ccbe\Proyecto_final\FILTRADO ORIGINAL"
RUTA_SALIDA = r"C:\Users\manue\Downloads\3ccbe\Proyecto_final\Filtracion 1\COMO QUEREMOS QUE SE LLAME"

df = pd.read_csv(RUTA_ENTRADA)

df_anfibios = df[df["class_name"] == "LO QUE QUEREMOS FILTRAR"].copy()

df_anfibios.to_csv(RUTA_SALIDA, index=False)

print(f"Registros originales : {len(df)}")
print(f"Registros anfibios   : {len(df_anfibios)}")
print(f"Archivo generado     : {RUTA_SALIDA}")

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models

from torch.utils.data import DataLoader, random_split

from DataLoaderFinal import PlantillaDataset

In [ ]:
import random

def set_seed(seed=1024):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
ruta_normalizado = r"CARACTERISTICAS"
ruta_importancia = r"RELIEFF // RANDOM FOREST"

dataset = PlantillaDataset(ruta_normalizado=ruta_normalizado,ruta_importancia=ruta_importancia, tamano_pixeles=128)

dataset_train, dataset_test = random_split(
dataset,[round(len(dataset)*0.7), len(dataset)-round(len(dataset)*0.7)],generator=torch.Generator().manual_seed(1024))

config = {
    "batch_size": 20,
    "num_epochs": 5,
    "num_hiddens": 64,
    "gamma_lr": 0.1,
    "learning_rate": 1e-3,
    "step_size": 4,
}

loader = DataLoader(dataset,batch_size=config["batch_size"],shuffle=True)

iterator = iter(loader)

training_loader = DataLoader(dataset_train,batch_size=config["batch_size"], shuffle=True)

test_loader = DataLoader(dataset_test,batch_size=config["batch_size"])

Codigo de Verificacion

In [ ]:
print("Número de clases:", dataset.n_classes)
print("Label mínimo:", dataset.df["label"].min())
print("Label máximo:", dataset.df["label"].max())
print("Labels únicos:", dataset.df["label"].nunique())

Verificacion de imagen con el Dataloader

In [ ]:
img = dataset._generar_imagen_pixel(dataset.df.iloc[0])
#imagen_np = np.flipud(np.fliplr(img.squeeze(0).numpy()))
imagen_np = np.flipud(img.squeeze(0).numpy())
plt.imshow(imagen_np, cmap="gray")
plt.axis("off")

In [ ]:
num_classes = dataset.n_classes

print(dataset.classes)
print("Número de clases:", num_classes)

weights = models.EfficientNet_B0_Weights.DEFAULT

model = models.efficientnet_b0(weights=weights)

# Cambiar entrada de 3 canales a 1 canal
conv_original = model.features[0][0]

model.features[0][0] = nn.Conv2d(
    in_channels=1,
    out_channels=conv_original.out_channels,
    kernel_size=conv_original.kernel_size,
    stride=conv_original.stride,
    padding=conv_original.padding,
    bias=False
)

# Inicializar usando el promedio de los 3 canales RGB
with torch.no_grad():
    model.features[0][0].weight[:] = conv_original.weight.mean(dim=1, keepdim=True)

# Congelar toda la red
for param in model.parameters():
    param.requires_grad = False

# Descongelar la primera convolución
for param in model.features[0][0].parameters():
    param.requires_grad = True

# Reemplazar clasificador
in_features = model.classifier[1].in_features

model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, num_classes)
)

# Entrenar el clasificador
for param in model.classifier.parameters():
    param.requires_grad = True

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

# Entrenamiento del modelo

In [ ]:
epochs = 15

for epoch in range(epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in training_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(training_loader)
    train_acc = 100 * correct / total

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)

        correct += (predicted==labels).sum().item()

print(f"Test Accuracy: {100*correct/total:.2f}%")

# Código para crear CSV de características con label encoding

In [ ]:
features = pd.read_csv("CARACT_NORMALIZADO.csv")
from sklearn.preprocessing import LabelEncoder

# Crear el encoder
label_encoder = LabelEncoder()

# Convertir la columna a etiquetas enteras
features["label"] = label_encoder.fit_transform(features["scientific_name"])
# Extraemos la columna 'label'
label_col = features['label']

# La eliminamos temporalmente del DataFrame
features = features.drop(columns=['label'])

# Insertamos la columna en la posición 2
features.insert(loc=2, column='label', value=label_col)

# features.to_csv("Caracteristicas.csv", index=False)

# Ver las correspondencias
# mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
# print(mapping)